# AWaRe Data Integration and Mapping
AWaRe is a classification published by the WHO to categorise antibiotics into Access, Watch, Reserve and Other groups.

The ["UK Access, Watch, Reserve, and Other classification for antibiotics"](https://www.gov.uk/government/publications/uk-aware-antibiotic-classification/uk-access-watch-reserve-and-other-classification-for-antibiotics-uk-aware-antibiotic-classification) is a UK adapted version of this.

The existing list on the UKHSA site is not machine readable. To facilitate research, it is helpful to link this categorisation to the [dm+d standard](https://www.nhsbsa.nhs.uk/pharmacies-gp-practices-and-appliance-contractors/dictionary-medicines-and-devices-dmd).

This notebook details a process to:
- Generate a dm+d-compatible AWaRe list using SQL-based logic.
- Compare this against a supplied UKHSA Excel version.
- Analyse differences, identify discrepancies, and reconcile them.

The work is divided into two main sections:
1. [**Mapping AWaRe using dm+d and SQL**](#s1)
2. [**Analysing UKHSA Excel-supplied data**](#s2)
3. [**Comparing UKHSA Excel-supplied data**](#s3)
4. [**Refining AWaRe dm+d list/SQL**](#s4)
5. [**Summary**](#s5)

### Imports
We need to import some libaries to help with the code

In [1]:
import pandas as pd
import requests
from ebmdatalab import bq
import os

<a id='s1'></a>
## Section 1: Mapping AWaRe to dm+d
This section uses SQL logic with dm+d data to extract a list of relevant VTMs and associated VMPs and map them to their AWaRe classification.

### Getting the AWaRe list
The AWaRe list is currently available on the following webpage. We can read the html page directly into Pandas to retrieve the first table.

In [2]:
# Set URL to scrape
url = "https://www.gov.uk/government/publications/uk-aware-antibiotic-classification/uk-access-watch-reserve-and-other-classification-for-antibiotics-uk-aware-antibiotic-classification"

# Read the first HTML table from the page into a DataFrame using the HTML content
df_scrape = pd.read_html(url)[0]

# Display the DataFrame
df_scrape

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category
0,Amikacin,Watch,Watch
1,Amoxicillin,Access,Access
2,Amoxicillin/ clavulanic-acid,Watch,Watch
3,Ampicillin,Access,Access
4,Azithromycin,Watch,Watch
...,...,...,...
85,Tigecycline,Reserve,Reserve
86,Tinidazole #,Other,Access
87,Tobramycin,Watch,Watch
88,Trimethoprim,Access,Access


### Cleaning up the list
The list contains some characters or identifiers that could break matching, we need to clean these up.

The # symbol is used to denote where category has changed, remove this

In [3]:
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace(" #", "", regex=False)

We can review remaining rows that contain non-alphabetical characters

In [4]:
df_scrape[df_scrape['Antibiotic'].str.contains(r'[^A-Za-z ]', na=False)]

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category
2,Amoxicillin/ clavulanic-acid,Watch,Watch
6,Benzathine-benzylpenicillin,Access,Access
20,Ceftaroline-fosamil,Reserve,Reserve
22,Ceftazidime/ avibactam,Reserve,Reserve
23,Ceftobiprole-medocaril,Reserve,Reserve
24,Ceftolozane/ tazobactam,Reserve,Reserve
31,"Colistin, intravenous",Reserve,Reserve
32,"Colistin, oral",Reserve,Reserve
34,Dalfopristin/ quinupristin,Reserve,Reserve
46,"Fosfomycin, intravenous",Reserve,Reserve


Some rows specify a route. Sometimes there is a comma between antibiotic and route, sometimes there isn't. We can pull any specified route into a seperate column.

In [5]:
# List of routes to check.
route_list = ["oral", "intravenous"]

# Function to extract the route if the Antibiotic entry ends with a route.
def extract_route(antibiotic):
    for route in route_list:
        if antibiotic.endswith(" " + route):
            return route
    return ""  # Return empty string if no route is found.

# Apply the function to create a new 'Route' column.
df_scrape['Route_specific'] = df_scrape['Antibiotic'].apply(extract_route)

# Remove any route names from the 'Antibiotic' column only if they appear at the end
# and are preceded by either ", " or " ".
for route in route_list:
    df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace(fr'(, | ){route}$', '', regex=True)


In [6]:
df_scrape[df_scrape['Antibiotic'].str.contains(r'[^A-Za-z ]', na=False)]

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category,Route_specific
2,Amoxicillin/ clavulanic-acid,Watch,Watch,
6,Benzathine-benzylpenicillin,Access,Access,
20,Ceftaroline-fosamil,Reserve,Reserve,
22,Ceftazidime/ avibactam,Reserve,Reserve,
23,Ceftobiprole-medocaril,Reserve,Reserve,
24,Ceftolozane/ tazobactam,Reserve,Reserve,
34,Dalfopristin/ quinupristin,Reserve,Reserve,
49,Imipenem/cilastatin,Reserve,Reserve,
50,Imipenem/cilastatin/relebactam,Reserve,Reserve,
55,Meropenem/ vaborbactam,Reserve,Reserve,


Some items contain a dash between words where as in dm+d there is a space. Replace the dash with a space to follow the common format.

In [7]:
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace("-", " ", regex=False)

Some items contain a forward slash between words to indicate a combination where as in dm+d there is a + symbol. Replace the dash with a + ensuring appropriately spaced to follow the common dm+d format.

In [8]:
# Replace / with a + ensuring it is appropriately spaced and 2nd drug name is capitalised
df_scrape['Antibiotic'] = (
    df_scrape['Antibiotic']
    .str.replace(r'\s*/\s*', ' + ', regex=True)
    .str.replace(r'(\+\s+)(\w)', lambda m: m.group(1) + m.group(2).upper(), regex=True)
)
# Create a new column 'Combo' that is True if 'Antibiotic' contains a plus between two words.
df_scrape['Combo'] = df_scrape['Antibiotic'].str.contains(r'\b\w+\s*\+\s*\w+\b', regex=True)

In [9]:
df_scrape[df_scrape['Antibiotic'].str.contains(r'[^A-Za-z ]', na=False)]

,Antibiotic,England-adapted 2019 AWaRe category,UK-adapted 2024 AWaRe category,Route_specific,Combo
2,Amoxicillin + Clavulanic acid,Watch,Watch,,True
22,Ceftazidime + Avibactam,Reserve,Reserve,,True
24,Ceftolozane + Tazobactam,Reserve,Reserve,,True
34,Dalfopristin + Quinupristin,Reserve,Reserve,,True
49,Imipenem + Cilastatin,Reserve,Reserve,,True
50,Imipenem + Cilastatin + Relebactam,Reserve,Reserve,,True
55,Meropenem + Vaborbactam,Reserve,Reserve,,True
69,Piperacillin + Tazobactam,Watch,Watch,,True
77,Sulfamethoxazole + Trimethoprim,Access,Access,,True


Some antibiotic combinations are referred to as different names or in a different order in dm+d. These can be identified via dm+d manual search. We need to convert the strings to match those used in dm+d.

In [10]:
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace("Amoxicillin + Clavulanic acid", "Co-amoxiclav", regex=False)
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace("Sulfamethoxazole + Trimethoprim", "Co-trimoxazole", regex=False)
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace("Dalfopristin + Quinupristin", "Quinupristin + Dalfopristin", regex=False)
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace("Imipenem + Cilastatin + Relebactam", "Cilastatin + Imipenem + Relebactam", regex=False)
df_scrape['Antibiotic'] = df_scrape['Antibiotic'].str.replace("Ceftobiprole medocaril", "Ceftobiprole", regex=False)

Add a column 'ATC_Route_specific' mapping 'Route_specific' to ATC route.  
Ensure 'Route_specific' is a string by replacing NaN with an empty string.  

In [11]:
# Define mapping
route_map = {'oral': 'O', 'intravenous': 'P'}

# Replace NaN with empty strings just to be safe
df_scrape['Route_specific'] = df_scrape['Route_specific'].fillna('')

# Map values using a dictionary, and fill any unmapped entries with empty string
df_scrape['ATC_Route_specific'] = df_scrape['Route_specific'].map(route_map).fillna('')

Reorder the columns and rename to simplify

In [12]:
df_scrape = df_scrape[['Antibiotic', 'Route_specific', 'ATC_Route_specific', 'Combo'] + [col for col in df_scrape.columns if col not in ['Antibiotic', 'Route_specific', 'ATC_Route_specific', 'Combo']]]

df_scrape.rename(columns={'England-adapted 2019 AWaRe category': 'aware_2019'}, inplace=True)
df_scrape.rename(columns={'UK-adapted 2024 AWaRe category': 'aware_2024'}, inplace=True)
df_scrape.rename(columns={'Route_specific': 'route'}, inplace=True)
df_scrape.rename(columns={'ATC_Route_specific': 'atc_route'}, inplace=True)

# Drop un-needed column
df_scrape.drop(columns='Combo', inplace=True)

In [13]:
df_prepared = df_scrape

with pd.option_context('display.max_rows', None):
    display(df_prepared)

,Antibiotic,route,atc_route,aware_2019,aware_2024
0,Amikacin,,,Watch,Watch
1,Amoxicillin,,,Access,Access
2,Co-amoxiclav,,,Watch,Watch
3,Ampicillin,,,Access,Access
4,Azithromycin,,,Watch,Watch
5,Aztreonam,,,Reserve,Reserve
6,Benzathine benzylpenicillin,,,Access,Access
7,Benzylpenicillin,,,Access,Access
8,Cefaclor,,,Watch,Watch
9,Cefadroxil,,,Watch,Access


We can save this as a CSV and upload as a table in BigQuery for further processing. `ebmdatalab.chris.aware_prepared` 

In [14]:
csv_path = os.path.join('data', 'AWaRe_prepared.csv')
df_prepared.to_csv(csv_path, index=False)

### Matching to VTMs

We can attempt to match Antibiotic names in the UKHSA to VTMs in dm+d.  

Firstly we will try directly matching.

In [15]:
sql = f"""
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
  FROM `ebmdatalab.chris.aware_list` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm = aware.Antibiotic
  WHERE vtm.invalid IS NOT TRUE
  ORDER BY Antibiotic ASC
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vtm_match.csv')

# Use the cached_read function from the bq library to run the query.
vtm_match = bq.cached_read(sql, csv_path=csv_path)

with pd.option_context('display.max_rows', None):
    display(vtm_match)

,Antibiotic,atc_route,aware_2019,aware_2024,nm
0,Amikacin,NaN,Watch,Watch,Amikacin
1,Amoxicillin,NaN,Access,Access,Amoxicillin
2,Ampicillin,NaN,Access,Access,Ampicillin
3,Azithromycin,NaN,Watch,Watch,Azithromycin
4,Aztreonam,NaN,Reserve,Reserve,Aztreonam
5,Benzathine benzylpenicillin,NaN,Access,Access,Benzathine benzylpenicillin
6,Benzylpenicillin,NaN,Access,Access,Benzylpenicillin
7,Cefaclor,NaN,Watch,Watch,Cefaclor
8,Cefadroxil,NaN,Watch,Access,Cefadroxil
9,Cefalexin,NaN,Watch,Access,Cefalexin


This works to match VTMs to many of the listing. But doesn't capture where there are multiple VTM matches - for example Amikacin and Amikacin liposomal for Amikacin. 

We can use more fuzzy matching to get greater sensitivity in the matches.

In [16]:
sql = f"""
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
  ORDER BY Antibiotic ASC
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vtm_fuzzy_match.csv')

# Use the cached_read function from the bq library to run the query.
vtm_fuzzy_match = bq.cached_read(sql, csv_path=csv_path)

with pd.option_context('display.max_rows', None):
    display(vtm_fuzzy_match)

,Antibiotic,atc_route,aware_2019,aware_2024,nm,match_type
0,Amikacin,NaN,Watch,Watch,Amikacin liposomal,ASSUMED
1,Amikacin,NaN,Watch,Watch,Amikacin,DIRECT
2,Amoxicillin,NaN,Access,Access,Amoxicillin,DIRECT
3,Ampicillin,NaN,Access,Access,Ampicillin,DIRECT
4,Ampicillin,NaN,Access,Access,Ampicillin + Flucloxacillin,ASSUMED
5,Azithromycin,NaN,Watch,Watch,Azithromycin,DIRECT
6,Aztreonam,NaN,Reserve,Reserve,Aztreonam + Avibactam,ASSUMED
7,Aztreonam,NaN,Reserve,Reserve,Aztreonam,DIRECT
8,Benzathine benzylpenicillin,NaN,Access,Access,Benzathine benzylpenicillin,DIRECT
9,Benzylpenicillin,NaN,Access,Access,Benzylpenicillin,DIRECT


This gives a number of duplicate matches - for example Meropenem + Vaborbactam matches to both Meropenem and Meropenem + Vaborbactam

We can look for all the duplicates

In [17]:
sql = f"""
WITH matched_vtms AS (
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
)

SELECT *
FROM matched_vtms
WHERE nm IN (
  SELECT nm
  FROM matched_vtms
  WHERE atc_route IS NULL
  GROUP BY nm
  HAVING COUNT(*) > 1
)
ORDER BY Antibiotic ASC;
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vtm_duplicates.csv')

# Use the cached_read function from the bq library to run the query.
vtm_duplicates = bq.cached_read(sql, csv_path=csv_path)

with pd.option_context('display.max_rows', None):
    display(vtm_duplicates)

,Antibiotic,atc_route,aware_2019,aware_2024,nm,match_type
0,Ampicillin,NaN,Access,Access,Ampicillin + Flucloxacillin,ASSUMED
1,Ceftazidime,NaN,Watch,Watch,Ceftazidime + Avibactam,ASSUMED
2,Ceftazidime + Avibactam,NaN,Reserve,Reserve,Ceftazidime + Avibactam,DIRECT
3,Flucloxacillin,NaN,Access,Access,Ampicillin + Flucloxacillin,ASSUMED
4,Meropenem,NaN,Reserve,Reserve,Meropenem + Vaborbactam,ASSUMED
5,Meropenem + Vaborbactam,NaN,Reserve,Reserve,Meropenem + Vaborbactam,DIRECT
6,Piperacillin,NaN,Watch,Watch,Piperacillin + Tazobactam,ASSUMED
7,Piperacillin + Tazobactam,NaN,Watch,Watch,Piperacillin + Tazobactam,DIRECT


From this we can create a list of the unwanted duplicates to remove from our full list.

In [18]:
sql = f"""
WITH matched_vtms AS (
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
)

SELECT *
FROM matched_vtms
WHERE nm IN (
  SELECT nm
  FROM matched_vtms
  WHERE atc_route IS NULL
  GROUP BY nm
  HAVING COUNT(*) > 1
) AND match_type = 'ASSUMED' 
    AND NOT (Antibiotic='Ampicillin' AND nm='Ampicillin + Flucloxacillin')
ORDER BY Antibiotic ASC;
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vtm_duplicates_direct.csv')

# Use the cached_read function from the bq library to run the query.
vtm_duplicates_direct = bq.cached_read(sql, csv_path=csv_path)

with pd.option_context('display.max_rows', None):
    display(vtm_duplicates_direct)

,Antibiotic,atc_route,aware_2019,aware_2024,nm,match_type
0,Ceftazidime,NaN,Watch,Watch,Ceftazidime + Avibactam,ASSUMED
1,Flucloxacillin,NaN,Access,Access,Ampicillin + Flucloxacillin,ASSUMED
2,Meropenem,NaN,Reserve,Reserve,Meropenem + Vaborbactam,ASSUMED
3,Piperacillin,NaN,Watch,Watch,Piperacillin + Tazobactam,ASSUMED


We can then create a full list of potential VTMs with duplicates removed.

In [19]:
sql = f"""
WITH matched_vtms AS ( -- This query does the fuzzy matching of Antibiotic name to VTM name
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    vtm.id,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
),

manual_exceptions AS ( -- This is a list of manual exceptions used when recluding assumed matches
  SELECT 'Ampicillin' AS Antibiotic, 'Ampicillin + Flucloxacillin' AS nm
  -- Add more rows here as needed
),

duplicate_vtms AS ( -- This produces a list of duplicate rows where a direct match already exists
  SELECT *
  FROM matched_vtms
  WHERE nm IN (
    SELECT nm
    FROM matched_vtms
    WHERE atc_route IS NULL
    GROUP BY nm
    HAVING COUNT(*) > 1
  )
  AND match_type = 'ASSUMED'
  AND NOT EXISTS (
    SELECT 1
    FROM manual_exceptions e
    WHERE e.Antibiotic = matched_vtms.Antibiotic AND e.nm = matched_vtms.nm
  )
)

SELECT
  matched_vtms.Antibiotic,
  matched_vtms.nm,
  matched_vtms.id,
  matched_vtms.atc_route,
  matched_vtms.aware_2019,
  matched_vtms.aware_2024,
FROM matched_vtms
WHERE NOT EXISTS ( -- removes the duplicates identified above
  SELECT 1
  FROM duplicate_vtms
  WHERE matched_vtms.Antibiotic = duplicate_vtms.Antibiotic AND matched_vtms.nm = duplicate_vtms.nm
)
ORDER BY matched_vtms.Antibiotic ASC ;
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vtm_match_cleaned.csv')

# Use the cached_read function from the bq library to run the query.
vtm_match_cleaned = bq.cached_read(sql, csv_path=csv_path)

with pd.option_context('display.max_rows', None):
    display(vtm_match_cleaned)

,Antibiotic,nm,id,atc_route,aware_2019,aware_2024
0,Amikacin,Amikacin liposomal,7.824210e+08,NaN,Watch,Watch
1,Amikacin,Amikacin,7.745340e+08,NaN,Watch,Watch
2,Amoxicillin,Amoxicillin,7.745860e+08,NaN,Access,Access
3,Ampicillin,Ampicillin + Flucloxacillin,7.745920e+08,NaN,Access,Access
4,Ampicillin,Ampicillin,7.745900e+08,NaN,Access,Access
5,Azithromycin,Azithromycin,7.747220e+08,NaN,Watch,Watch
6,Aztreonam,Aztreonam + Avibactam,4.379671e+16,NaN,Reserve,Reserve
7,Aztreonam,Aztreonam,7.747250e+08,NaN,Reserve,Reserve
8,Benzathine benzylpenicillin,Benzathine benzylpenicillin,1.234764e+09,NaN,Access,Access
9,Benzylpenicillin,Benzylpenicillin,7.748260e+08,NaN,Access,Access


In [20]:
vtm_match_cleaned.to_csv("data/vtm_match_clean.csv", index=False)

### Bringing in VMPs

Now that we have a list of dm+d compatible VTMs, we can bring in matching VMPs.

In [21]:
sql = f"""
WITH matched_vtms AS ( -- This query does the fuzzy matching of Antibiotic name to VTM name
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    vtm.id,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
),

manual_exceptions AS ( -- This is a list of manual exceptions used when recluding assumed matches
  SELECT 'Ampicillin' AS Antibiotic, 'Ampicillin + Flucloxacillin' AS nm
  -- Add more rows here as needed
),

duplicate_vtms AS ( -- This produces a list of duplicate rows where a direct match already exists
  SELECT *
  FROM matched_vtms
  WHERE nm IN (
    SELECT nm
    FROM matched_vtms
    WHERE atc_route IS NULL
    GROUP BY nm
    HAVING COUNT(*) > 1
  )
  AND match_type = 'ASSUMED'
  AND NOT EXISTS (
    SELECT 1
    FROM manual_exceptions e
    WHERE e.Antibiotic = matched_vtms.Antibiotic AND e.nm = matched_vtms.nm
  )
),

vmp_with_route AS (
  SELECT DISTINCT
    vmp.id AS id,
    vmp.vtm AS vtm,
    vmp.nm AS nm,
    COALESCE(sroute.route, routelookup.who_route) AS vmp_atc_route
  FROM `ebmdatalab.dmd.vmp` vmp
  LEFT JOIN dmd.ont ont ON vmp.id = ont.vmp
  LEFT JOIN dmd.ontformroute ofr ON ont.form = ofr.cd
  LEFT JOIN chris.vmp_single_route_identifier sroute ON vmp.id = sroute.vmp_id
  LEFT JOIN `scmd_dmd_views.dmd_to_atc_route` routelookup ON ofr.descr = routelookup.dmd_ofr
)

SELECT DISTINCT
  matched_vtms.Antibiotic,
  matched_vtms.nm as vtm_nm,
  matched_vtms.id as vtm_id,
  vmp.nm as vmp_nm,
  vmp.id as vmp_id,
  vmp.vmp_atc_route,
  matched_vtms.atc_route,
  matched_vtms.aware_2019,
  matched_vtms.aware_2024
FROM matched_vtms
LEFT JOIN vmp_with_route vmp
  ON matched_vtms.id = vmp.vtm
  AND (
    matched_vtms.atc_route IS NULL
    OR matched_vtms.atc_route = vmp.vmp_atc_route
  )
WHERE NOT EXISTS ( -- removes the duplicates identified above
  SELECT 1
  FROM duplicate_vtms
  WHERE matched_vtms.Antibiotic = duplicate_vtms.Antibiotic AND matched_vtms.nm = duplicate_vtms.nm
)
ORDER BY matched_vtms.Antibiotic ASC;
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vmps_matched.csv')

# Use the cached_read function from the bq library to run the query.
vmps_matched = bq.cached_read(sql, csv_path=csv_path)

vmps_matched

Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


,Antibiotic,vtm_nm,vtm_id,vmp_nm,vmp_id,vmp_atc_route,atc_route,aware_2019,aware_2024
0,Amikacin,Amikacin,774534000,Amikacin 500mg/2ml solution for injection ampo...,44422511000001103,P,None,Watch,Watch
1,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection ampoules,44007311000001107,None,None,Watch,Watch
2,Amikacin,Amikacin,774534000,Amikacin 500mg/100ml infusion polyethylene bot...,41823511000001108,P,None,Watch,Watch
3,Amikacin,Amikacin,774534000,Amikacin 400micrograms/0.1ml intravitreal inje...,38244211000001108,None,None,Watch,Watch
4,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection vials,35111911000001104,None,None,Watch,Watch
...,...,...,...,...,...,...,...,...,...
703,Vancomycin,Vancomycin,777914002,Vancomycin 62.5mg/5ml oral solution,13455811000001106,O,None,Watch,Watch
704,Vancomycin,Vancomycin,777914002,Vancomycin 125mg/5ml oral solution,15453011000001107,O,None,Watch,Watch
705,Vancomycin,Vancomycin,777914002,Vancomycin 125mg/5ml oral suspension,15453111000001108,O,None,Watch,Watch
706,Vancomycin,Vancomycin,777914002,Vancomycin 5% eye drops,21407511000001100,None,None,Watch,Watch


Inspecting this list we can see a number of non-systemic antibiotics which are not considered as part of the AWaRe classification - e.g. eye drops, ointments etc. We can remove these by filtering out when selecting VMPs.

In [22]:
sql = f"""
WITH matched_vtms AS ( -- This query does the fuzzy matching of Antibiotic name to VTM name
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    vtm.id,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
),

manual_exceptions AS ( -- This is a list of manual exceptions used when recluding assumed matches
  SELECT 'Ampicillin' AS Antibiotic, 'Ampicillin + Flucloxacillin' AS nm
  -- Add more rows here as needed
),

duplicate_vtms AS ( -- This produces a list of duplicate rows where a direct match already exists
  SELECT *
  FROM matched_vtms
  WHERE nm IN (
    SELECT nm
    FROM matched_vtms
    WHERE atc_route IS NULL
    GROUP BY nm
    HAVING COUNT(*) > 1
  )
  AND match_type = 'ASSUMED'
  AND NOT EXISTS (
    SELECT 1
    FROM manual_exceptions e
    WHERE e.Antibiotic = matched_vtms.Antibiotic AND e.nm = matched_vtms.nm
  )
),

vmp_with_route AS (
  SELECT DISTINCT
    vmp.id AS id,
    vmp.vtm AS vtm,
    vmp.nm AS nm,
    COALESCE(sroute.route, routelookup.who_route) AS vmp_atc_route
  FROM `ebmdatalab.dmd.vmp` vmp
  LEFT JOIN dmd.ont ont ON vmp.id = ont.vmp
  LEFT JOIN dmd.ontformroute ofr ON ont.form = ofr.cd
  LEFT JOIN chris.vmp_single_route_identifier sroute ON vmp.id = sroute.vmp_id
  LEFT JOIN `scmd_dmd_views.dmd_to_atc_route` routelookup ON ofr.descr = routelookup.dmd_ofr
  WHERE ofr.descr NOT LIKE '%.auricular'
  AND ofr.descr NOT LIKE '%.cutaneous'
  AND ofr.descr NOT LIKE '%.ophthalmic'
  AND ofr.descr NOT LIKE '%.oromucosal'
  AND ofr.descr NOT LIKE '%.intralesional'
  AND ofr.descr NOT LIKE '%.nasal'
)

SELECT DISTINCT
  matched_vtms.Antibiotic,
  matched_vtms.nm as vtm_nm,
  matched_vtms.id as vtm_id,
  vmp.nm as vmp_nm,
  vmp.id as vmp_id,
  vmp.vmp_atc_route,
  matched_vtms.atc_route,
  matched_vtms.aware_2019,
  matched_vtms.aware_2024
FROM matched_vtms
LEFT JOIN vmp_with_route vmp
  ON matched_vtms.id = vmp.vtm
  AND (
    matched_vtms.atc_route IS NULL
    OR matched_vtms.atc_route = vmp.vmp_atc_route
  )
WHERE NOT EXISTS ( -- removes the duplicates identified above
  SELECT 1
  FROM duplicate_vtms
  WHERE matched_vtms.Antibiotic = duplicate_vtms.Antibiotic AND matched_vtms.nm = duplicate_vtms.nm
) AND vmp.id IS NOT NULL
ORDER BY matched_vtms.Antibiotic ASC;
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vmps_matched.csv')

# Use the cached_read function from the bq library to run the query.
vmps_matched = bq.cached_read(sql, csv_path=csv_path)

with pd.option_context('display.max_rows', None):
    display(vmps_matched)

Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


,Antibiotic,vtm_nm,vtm_id,vmp_nm,vmp_id,vmp_atc_route,atc_route,aware_2019,aware_2024
0,Amikacin,Amikacin,774534000,Amikacin 500mg/2ml solution for injection ampo...,44422511000001103,P,None,Watch,Watch
1,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection ampoules,44007311000001107,None,None,Watch,Watch
2,Amikacin,Amikacin,774534000,Amikacin 500mg/100ml infusion polyethylene bot...,41823511000001108,P,None,Watch,Watch
3,Amikacin,Amikacin,774534000,Amikacin 400micrograms/0.1ml intravitreal inje...,38244211000001108,None,None,Watch,Watch
4,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection vials,35111911000001104,None,None,Watch,Watch
5,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection pre-f...,35104511000001105,None,None,Watch,Watch
6,Amikacin,Amikacin,774534000,Amikacin 500mg/2ml solution for injection vials,35899911000001109,P,None,Watch,Watch
7,Amikacin,Amikacin,774534000,Amikacin 100mg/2ml solution for injection vials,35899811000001104,P,None,Watch,Watch
8,Amikacin,Amikacin liposomal,782421008,Amikacin liposomal 590mg nebuliser dispersion ...,39601711000001109,Inhal.solution,None,Watch,Watch
9,Amoxicillin,Amoxicillin,774586009,Amoxicillin 500mg/50ml infusion bags,34226911000001109,P,None,Access,Access


In [23]:
vmps_matched.to_csv('data/vtm_method.csv', index=False)

<a id='s2'></a>
## Section 2: Processing and Comparing AWaRe Excel Data
This section loads and analyses the AWaRe Excel data provided by UKHSA. It compares the content with the previously generated VTM-based method and highlights any discrepancies.

### Getting the AWaRe list
The AWaRe list has been provided by the UKHSA team as an Excel spreadsheet. We can save the Excel spreadsheet as a CSV file then read into Pandas.

In [24]:
excel_path = os.path.join('..', 'data', 'AWaRe_UK_2024_translation_table_FINAL_UPDATED_20250429.xlsx')
df_ukhsa = pd.read_excel(excel_path)
df_ukhsa

,dmd,Antibiotic,Class,atccode,route,VmpUnit,WHO_AWaRe_2023,UK_AWaRe_2024,Include,Notes
0,Amikacin 100mg/2ml solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
1,Amikacin 1165mg/100ml solution for injection v...,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
2,Amikacin 1200mg solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
3,Amikacin 1300mg/50ml solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
4,Amikacin 250mg/2ml injection,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN
...,...,...,...,...,...,...,...,...,...,...
926,Vancomycin 50mg/5ml oral solution,Vancomycin_oral,Glycopeptides,A07AA09,O,ml,w,w,1,NaN
927,Vancomycin 5mg/0.5ml solution for injection pr...,Vancomycin_IV,Glycopeptides,J01XA01,P,ml,w,w,1,NaN
928,Vancomycin 62.5mg/5ml oral solution,Vancomycin_oral,Glycopeptides,A07AA09,O,ml,w,w,1,NaN
929,Vancomycin 750mg/250ml infusion bags,Vancomycin_IV,Glycopeptides,J01XA01,P,ml,w,w,1,NaN


Extract data for VMPs from dm+d tables held in BigQuery

In [25]:
# Construct the query using the formatted list.
sql = f"""
SELECT DISTINCT
  vmp.id AS vmp_id,
  vmp.nm AS vmp_nm,
  vmp.invalid AS invalid_vmp,
  vtm.id AS vtm_id,
  vtm.nm AS vtm_nm,
  COALESCE(sroute.route, routelookup.who_route) AS atc_route,
  atclookup.ATC AS atc
FROM `ebmdatalab.dmd.vmp` vmp
LEFT JOIN dmd.ont ont ON vmp.id = ont.vmp
LEFT JOIN dmd.vtm vtm ON vmp.vtm = vtm.id
LEFT JOIN dmd.ontformroute ofr ON ont.form = ofr.cd
LEFT JOIN `dmd.vpi_to_atc_and_ddd` atclookup ON vmp.id = atclookup.VPID
LEFT JOIN chris.vmp_single_route_identifier sroute ON vmp.id = sroute.vmp_id -- this table maps VMP to a specific route where more than one is given
LEFT JOIN `scmd_dmd_views.dmd_to_atc_route` routelookup ON ofr.descr = routelookup.dmd_ofr  -- this table maps dmd route to ATC route
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vmp_vtms.csv')

# Use the cached_read function from the bq library to run the query.
vmp_vtms = bq.cached_read(sql, csv_path=csv_path)

# Force vmp_id and vtm_id to proper string type
vmp_vtms['vmp_id'] = vmp_vtms['vmp_id'].astype('string')
vmp_vtms['vtm_id'] = vmp_vtms['vtm_id'].astype('string')

In [26]:
vmp_vtms

,vmp_id,vmp_nm,invalid_vmp,vtm_id,vtm_nm,atc_route,atc
0,45010811000001105,Lazertinib 80mg tablets,False,4.49914110000011e+16,Lazertinib,O,NaN
1,44117211000001108,Generic DryMax Super dressing 5cm x 5cm square,False,<NA>,NaN,NaN,NaN
2,42801911000001108,Catheter safety valves,False,<NA>,NaN,NaN,NaN
3,41960011000001106,Copper coated barrier dressing sterile with ad...,False,<NA>,NaN,NaN,NaN
4,41535211000001108,Odevixibat 600microgram capsules,False,1179275005.0,Odevixibat,O,A05AX05
...,...,...,...,...,...,...,...
24698,43907811000001106,Somapacitan 10mg/1.5ml solution for injection ...,False,898161005.0,Somapacitan,P,NaN
24699,43907911000001101,Somapacitan 15mg/1.5ml solution for injection ...,False,898161005.0,Somapacitan,P,NaN
24700,44098211000001102,Budesonide 1mg/2ml nebuliser suspension unit d...,False,774924004.0,Budesonide,Inhal.solution,NaN
24701,43813811000001108,Generic PDE reach5 oral powder 18g sachets,False,<NA>,NaN,O,NaN


Attempt to match VMPs to UKHSA table

In [27]:
# Ensure IDs are strings before merging
vmp_vtms['vmp_id'] = vmp_vtms['vmp_id'].astype('string')
vmp_vtms['vtm_id'] = vmp_vtms['vtm_id'].astype('string')

# Add lower-case name columns for matching
df_ukhsa['dmd_lower'] = df_ukhsa['dmd'].str.lower()
vmp_vtms['vmp_nm_lower'] = vmp_vtms['vmp_nm'].str.lower()

# Merge on lower-case names
ukhsa_original_matched = (
    df_ukhsa.merge(
        vmp_vtms,
        left_on='dmd_lower',
        right_on='vmp_nm_lower',
        how='left'
    )
    .drop(columns=['dmd_lower', 'vmp_nm_lower'])
)

ukhsa_original_matched

,dmd,Antibiotic,Class,atccode,route,VmpUnit,WHO_AWaRe_2023,UK_AWaRe_2024,Include,Notes,vmp_id,vmp_nm,invalid_vmp,vtm_id,vtm_nm,atc_route,atc
0,Amikacin 100mg/2ml solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN,35899811000001104,Amikacin 100mg/2ml solution for injection vials,False,774534000.0,Amikacin,P,J01GB06
1,Amikacin 1165mg/100ml solution for injection v...,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
2,Amikacin 1200mg solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
3,Amikacin 1300mg/50ml solution for injection vials,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
4,Amikacin 250mg/2ml injection,Amikacin,Aminoglycosides,J01GB06,P,ml,a,w,1,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,Vancomycin 50mg/5ml oral solution,Vancomycin_oral,Glycopeptides,A07AA09,O,ml,w,w,1,NaN,12940811000001109,Vancomycin 50mg/5ml oral solution,False,777914002.0,Vancomycin,O,A07AA09
944,Vancomycin 5mg/0.5ml solution for injection pr...,Vancomycin_IV,Glycopeptides,J01XA01,P,ml,w,w,1,NaN,21278311000001101,Vancomycin 5mg/0.5ml solution for injection pr...,False,777914002.0,Vancomycin,NaN,J01XA01
945,Vancomycin 62.5mg/5ml oral solution,Vancomycin_oral,Glycopeptides,A07AA09,O,ml,w,w,1,NaN,13455811000001106,Vancomycin 62.5mg/5ml oral solution,False,777914002.0,Vancomycin,O,A07AA09
946,Vancomycin 750mg/250ml infusion bags,Vancomycin_IV,Glycopeptides,J01XA01,P,ml,w,w,1,NaN,35051211000001109,Vancomycin 750mg/250ml infusion bags,False,777914002.0,Vancomycin,P,J01XA01


In [28]:
ukhsa_original_matched.to_csv("ukhsa_original_matched.csv")

In [29]:
ukhsa_original_matched_analysis=ukhsa_original_matched

In [30]:
no_vmp_id_count = ukhsa_original_matched_analysis[ukhsa_original_matched_analysis['vmp_id'].isna() | (ukhsa_original_matched_analysis['vmp_id'] == 0)].shape[0]
print(f"Rows without a matched vmp_id: {no_vmp_id_count}")

Rows without a matched vmp_id: 186


In [31]:
# Filter rows where vmp_id is NA or 0
rows_without_vmpid = ukhsa_original_matched_analysis[
    ukhsa_original_matched_analysis['vmp_id'].isna() | 
    (ukhsa_original_matched_analysis['vmp_id'] == 0)
]

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(rows_without_vmpid[['dmd', 'Antibiotic', 'Include']])

,dmd,Antibiotic,Include
1,Amikacin 1165mg/100ml solution for injection vials,Amikacin,1
2,Amikacin 1200mg solution for injection vials,Amikacin,1
3,Amikacin 1300mg/50ml solution for injection vials,Amikacin,1
4,Amikacin 250mg/2ml injection,Amikacin,1
39,Aztreonam 3g powder for solution for injection vials,Aztreonam,1
48,Benzathine benzylpenicillin 600000unit powder and solvent for suspension for injection vials,Benzathine-benzylpenicillin,1
49,Benzathine benzylpenicllin 900mg injection,Benzathine-benzylpenicillin,1
50,Benzylpenicillin 1.2g powder for solution for injection vials,Benzylpenicillin,1
51,Benzylpenicillin 1.2g/50ml infusion bags,Benzylpenicillin,1
52,Benzylpenicillin 2.4g/100ml infusion bags,Benzylpenicillin,1


From a manual review these are the rows from the above which have potential matches - but may not have matched above due to variations in wording used.

In [32]:

selected_ids = [
    48, 49, 50, 51, 53, 54, 164, 173, 178, 181, 228, 241, 269, 286, 290,
    306, 310, 315, 317, 340, 372, 373, 385, 389, 390, 408, 474, 475, 502,
    504, 506, 530, 672, 676, 677, 681, 692, 694, 765, 793, 831, 835, 851,
    900, 909, 912, 917, 935, 936
]

# Filter rows_without_vmpid using those index numbers
selected_rows = rows_without_vmpid.loc[rows_without_vmpid.index.isin(selected_ids)]

# Display the selected rows with full column width
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(selected_rows[['dmd', 'Antibiotic', 'Include']])

,dmd,Antibiotic,Include
48,Benzathine benzylpenicillin 600000unit powder and solvent for suspension for injection vials,Benzathine-benzylpenicillin,1
49,Benzathine benzylpenicllin 900mg injection,Benzathine-benzylpenicillin,1
50,Benzylpenicillin 1.2g powder for solution for injection vials,Benzylpenicillin,1
51,Benzylpenicillin 1.2g/50ml infusion bags,Benzylpenicillin,1
53,Benzylpenicillin 600mg powder for solution for injection vials,Benzylpenicillin,1
54,Benzylpenicillin 600mg/50ml infusion bags,Benzylpenicillin,1
164,Cefuroxime 1.5g powder for injection vials,Cefuroxime,1
173,Cefuroxime 250mg powder for injection vials,Cefuroxime,1
178,Cefuroxime 3mg in 0.3ml solution for injection pre-filled syringes,Cefuroxime,1
181,Cefuroxime 750mg powder for injection vials,Cefuroxime,1


Matching all products under a specific VTM may give a more systematic approach to creating/updating this list in dm+d. We can check for how many match to a VMP but miss a VTM ID.

In [33]:
no_vtm_id_count = ukhsa_original_matched_analysis[
    (ukhsa_original_matched_analysis['vmp_id'].notna()) &
    (ukhsa_original_matched_analysis['vmp_id'] != 0) &
    (ukhsa_original_matched_analysis['vtm_id'].isna())
].shape[0]

print(f"Rows with a vmp_id but missing vtm_id: {no_vtm_id_count}")

Rows with a vmp_id but missing vtm_id: 3


In [34]:

# Filter rows where vmp_id is present but vtm_id is missing
filtered_rows = ukhsa_original_matched_analysis[
    (ukhsa_original_matched_analysis['vmp_id'].notna()) &
    (ukhsa_original_matched_analysis['vmp_id'] != 0) &
    (ukhsa_original_matched_analysis['vtm_id'].isna())
]

# Display the filtered rows
filtered_rows

,dmd,Antibiotic,Class,atccode,route,VmpUnit,WHO_AWaRe_2023,UK_AWaRe_2024,Include,Notes,vmp_id,vmp_nm,invalid_vmp,vtm_id,vtm_nm,atc_route,atc
371,Flucloxacillin 125mg / 5ml oral solution sugar...,Flucloxacillin,Penicillins,J01CF05,O,ml,a,a,1,NaN,7526011000001106,Flucloxacillin 125mg / 5ml oral solution sugar...,True,<NA>,NaN,O,NaN
413,Generic deteclo 300mg tablets,Chlortetracycline hydrochloride/demeclocycline...,Tetracyclines,J01AA20,O,tablet,o,o,0,"excl, discontinued in UK since 2007",34819711000001101,Generic Deteclo 300mg tablets,False,<NA>,NaN,O,J01AA20
415,Generic voractiv tablets,Ethambutol hydrochloride/isoniazid/pyrazinamid...,Antimycobacterials,J04AM06,O,tablet,o,o,0,anti-TB,21733111000001101,Generic Voractiv tablets,False,<NA>,NaN,O,J04AM06


Of these results, "Flucloxacillin 125mg / 5ml oral solution sugar free" is an invalid VMP in dm+d so can be disregareded. The other 2 products are combination treatments for TB and are excluded from the AWaRe list.

In [35]:
grouped_vtm_include = ukhsa_original_matched_analysis.groupby(['vtm_nm', 'Include'], dropna=False).size().reset_index(name='count')

# Display the selected rows with full column width
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(grouped_vtm_include)

,vtm_nm,Include,count
0,Amikacin,1,5
1,Amikacin liposomal,1,1
2,Amoxicillin,1,14
3,Ampicillin,1,6
4,Azithromycin,1,6
5,Aztreonam,1,5
6,Aztreonam + Avibactam,1,1
7,Bedaquiline,0,1
8,Benzathine benzylpenicillin,1,4
9,Benzylpenicillin,1,4


In this list only 1 VTM has rows considered both included and excluded.

In [36]:
tobramycin_rows = ukhsa_original_matched_analysis[
    ukhsa_original_matched_analysis['vtm_nm'] == 'Tobramycin'
]

tobramycin_rows

,dmd,Antibiotic,Class,atccode,route,VmpUnit,WHO_AWaRe_2023,UK_AWaRe_2024,Include,Notes,vmp_id,vmp_nm,invalid_vmp,vtm_id,vtm_nm,atc_route,atc
838,Tobramycin 135mg/5ml oral solution,Tobramycin,Aminoglycosides,J01GB01,O,ml,w,w,0,no WHO DDD,21718511000001106,Tobramycin 135mg/5ml oral solution,False,777791002.0,Tobramycin,O,NaN
841,Tobramycin 170mg/1.7ml nebuliser liquid ampoules,Tobramycin,Aminoglycosides,J01GB01,INH,ml,w,w,1,NaN,32936811000001108,Tobramycin 170mg/1.7ml nebuliser liquid ampoules,False,777791002.0,Tobramycin,Inhal.solution,J01GB01
844,Tobramycin 200mg/5ml oral solution,Tobramycin,Aminoglycosides,J01GB01,O,ml,w,w,0,no WHO DDD,35084911000001108,Tobramycin 200mg/5ml oral solution,False,777791002.0,Tobramycin,O,NaN
845,Tobramycin 20mg/2ml solution for injection vials,Tobramycin,Aminoglycosides,J01GB01,P,ml,w,w,1,NaN,34194211000001103,Tobramycin 20mg/2ml solution for injection vials,False,777791002.0,Tobramycin,P,J01GB01
846,Tobramycin 20mg/5ml oral solution,Tobramycin,Aminoglycosides,J01GB01,O,ml,w,w,0,no WHO DDD,23676411000001103,Tobramycin 20mg/5ml oral solution,False,777791002.0,Tobramycin,O,J01GB01
853,Tobramycin 240mg/6ml solution for injection vials,Tobramycin,Aminoglycosides,J01GB01,P,ml,w,w,1,NaN,4516011000001106,Tobramycin 240mg/6ml solution for injection vials,False,777791002.0,Tobramycin,P,J01GB01
859,Tobramycin 28mg inhalation powder capsules wit...,Tobramycin,Aminoglycosides,J01GB01,INH,capsule,w,w,1,NaN,19544811000001101,Tobramycin 28mg inhalation powder capsules wit...,False,777791002.0,Tobramycin,Inhal.powder,J01GB01
862,Tobramycin 300mg/4ml nebuliser liquid ampoules,Tobramycin,Aminoglycosides,J01GB01,INH,ml,w,w,1,NaN,14801411000001105,Tobramycin 300mg/4ml nebuliser liquid ampoules,False,777791002.0,Tobramycin,Inhal.solution,J01GB01
863,Tobramycin 300mg/5ml nebuliser liquid ampoules,Tobramycin,Aminoglycosides,J01GB01,INH,ml,w,w,1,NaN,4145111000001102,Tobramycin 300mg/5ml nebuliser liquid ampoules,False,777791002.0,Tobramycin,Inhal.solution,J01GB01
872,Tobramycin 400mg/5ml oral solution,Tobramycin,Aminoglycosides,J01GB01,O,ml,w,w,0,no WHO DDD,21717911000001100,Tobramycin 400mg/5ml oral solution,False,777791002.0,Tobramycin,O,NaN


Looking at the rows we can see oral tobramycin has been excluded as there is no WHO DDD. We will keep in for now but in later code can exclude any VMPs without a DDD if needed.

<a id='s3'></a>
## Section 3: Validate dm+d list/SQL method

We have generated a list of VMPs by matching VTMs to UKHSA descriptions published online. We want to compare the results of this with the provided data.

In [37]:
vtm_method = pd.read_csv("vtm_method.csv", dtype={'vmp_id': 'string', 'vtm_id': 'string'})
vtm_method

,Antibiotic,vtm_nm,vtm_id,vmp_nm,vmp_id,vmp_atc_route,atc_route,aware_2019,aware_2024
0,Amikacin,Amikacin,774534000,Amikacin 500mg/2ml solution for injection ampo...,44422511000001103,P,NaN,Watch,Watch
1,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection ampoules,44007311000001107,NaN,NaN,Watch,Watch
2,Amikacin,Amikacin,774534000,Amikacin 500mg/100ml infusion polyethylene bot...,41823511000001108,P,NaN,Watch,Watch
3,Amikacin,Amikacin,774534000,Amikacin 400micrograms/0.1ml intravitreal inje...,38244211000001108,NaN,NaN,Watch,Watch
4,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection vials,35111911000001104,NaN,NaN,Watch,Watch
...,...,...,...,...,...,...,...,...,...
591,Vancomycin,Vancomycin,777914002,Vancomycin 250mg/5ml oral suspension,8818711000001109,O,NaN,Watch,Watch
592,Vancomycin,Vancomycin,777914002,Vancomycin 50mg/5ml oral solution,12940811000001109,O,NaN,Watch,Watch
593,Vancomycin,Vancomycin,777914002,Vancomycin 62.5mg/5ml oral solution,13455811000001106,O,NaN,Watch,Watch
594,Vancomycin,Vancomycin,777914002,Vancomycin 125mg/5ml oral solution,15453011000001107,O,NaN,Watch,Watch


The VTM/VMP methodology should allow us to accurately identify all appropriate VMPs assuming we have a matching list of VTMs.  

We can compare the list of VTMs in our methodology against the list of VTMs in the UKHSA list to ensure we have identified the list correctly.

In [38]:
ukhsa_original_matched_analysis_VTM = ukhsa_original_matched

Firstly get a list of unique VTMs in the UKHSA list where Include = 1 - i.e. where they are considered as part of the AWaRe categorisation.

In [39]:
# Step 1: Get full unique VTM rows with AWaRe columns where Include == 1
ukhsa_unique_vtm_aware_included = ukhsa_original_matched_analysis.loc[
    ukhsa_original_matched_analysis['Include'] == 1,
    ['vtm_nm', 'route', 'WHO_AWaRe_2023', 'UK_AWaRe_2024']
].dropna(subset=['vtm_nm']).drop_duplicates()

# Step 2: Extract just the vtm_nm column without duplicates
ukhsa_unique_vtm_included = ukhsa_unique_vtm_aware_included['vtm_nm'].drop_duplicates()
ukhsa_unique_vtm_included

0                           Amikacin
9                 Amikacin liposomal
10                       Amoxicillin
24                        Ampicillin
30                      Azithromycin
                   ...              
829    Ticarcillin + Clavulanic acid
832                      Tigecycline
841                       Tobramycin
891                     Trimethoprim
904                       Vancomycin
Name: vtm_nm, Length: 93, dtype: object

Get a list of unique VTMs from the VTM method list.

In [40]:
vtm_method_unique_vtm = vtm_method.loc[:, 'vtm_nm'].dropna().drop_duplicates()
vtm_method_unique_vtm

0                Amikacin
8      Amikacin liposomal
9             Amoxicillin
24             Ampicillin
30           Azithromycin
              ...        
536           Tigecycline
537            Tinidazole
538            Tobramycin
553          Trimethoprim
565            Vancomycin
Name: vtm_nm, Length: 91, dtype: object

The UKHSA provided dataset has 93 unique VTMs. Our generated VTM list only contains 91. We can find the discrepancies.

In [41]:
# Start with full UKHSA DataFrame (including AWaRe columns)
ukhsa_df = ukhsa_unique_vtm_aware_included[['vtm_nm', 'WHO_AWaRe_2023', 'UK_AWaRe_2024']]

# Convert vtm_method list to DataFrame
vtm_method_df = pd.DataFrame({'vtm_nm': vtm_method_unique_vtm})

# Merge with indicator
merged_vtm = pd.merge(
    ukhsa_df,
    vtm_method_df,
    on='vtm_nm',
    how='outer',
    indicator=True
)

# Rename categories of the _merge column
merged_vtm['_merge'] = merged_vtm['_merge'].cat.rename_categories({
    'left_only': 'ukhsa_provided_only',
    'right_only': 'vtm_method_only',
    'both': 'both'
})

# Filter out matches if you only want differences
vtm_differences = merged_vtm[merged_vtm['_merge'] != 'both']

# Sort by source
vtm_differences = vtm_differences.sort_values(by='_merge')

# Result: vtm_differences includes AWaRe columns where available
vtm_differences.drop_duplicates()


,vtm_nm,WHO_AWaRe_2023,UK_AWaRe_2024,_merge
24,Cefpirome,w,r,ukhsa_provided_only
25,Cefpodoxime,w,w,ukhsa_provided_only
26,Cefprozil,w,w,ukhsa_provided_only
48,Co-fluampicil,NR,o,ukhsa_provided_only
91,Nalidixic acid,o,o,ukhsa_provided_only
122,Ticarcillin + Clavulanic acid,o,o,ukhsa_provided_only
93,Netilmicin,w,o,ukhsa_provided_only
106,Roxithromycin,w,o,ukhsa_provided_only
107,Sodium fusidate,w,w,ukhsa_provided_only
19,Cefepime + Enmetazobactam,NaN,NaN,vtm_method_only


Reviewing the results above.  
**ukhsa_provided_only**
- Cefpirome, cefpodoxime, cefprozil, nalidixic acid, ticarcillin + clavulanic acid, netilmicin, roxithromycin - not listed on AWaRe list on UKHSA website. All appear either not in use or rarely used.
- Co-fluampicil - we have this in the VTM method list under ampicillin + flucloxacillin - it's unclear why there are 2 seperate VTMs for apparently the same thing but possibly in part due to neither VTM having an active VMP. We can classify both with the category 'Other'.
- Sodium fusidate is covered by fusidic acid listed in the original - we need to add the specific VTM sodium fusidate.

**vtm_method_only**
- Cefepime + Enmetazobactam	- has been identified due to cefepime component.
- Telavancin, tinidazole, neomycin - all listed in the online UKHSA list
- Spiramycin, demeclocycline, spectinomycin - listed in the online UKHSA list under 'Other'

<a id='s4'></a>
## Section 4: Extend/refine the SQL/VTM method
This section expands the SQL logic in section 1 to add in the additional VTMs identified in section 3.

Get the original list of VTMs and extend with the additional VTMS identified in section 3.

In [42]:
vtm_match_cleaned = pd.read_csv("vtm_match_clean.csv")

# Step 1: Define manually added unique rows with full AWaRe labels
vtm_manual_additions = [
    {'Antibiotic': 'Cefpirome', 'aware_2019': 'Watch', 'aware_2024': 'Reserve'},
    {'Antibiotic': 'Cefpodoxime', 'aware_2019': 'Watch', 'aware_2024': 'Watch'},
    {'Antibiotic': 'Cefprozil', 'aware_2019': 'Watch', 'aware_2024': 'Watch'},
    {'Antibiotic': 'Co-fluampicil', 'aware_2019': 'Other', 'aware_2024': 'Other'},
    {'Antibiotic': 'Nalidixic acid', 'aware_2019': 'Other', 'aware_2024': 'Other'},
    {'Antibiotic': 'Ticarcillin + Clavulanic acid', 'aware_2019': 'Other', 'aware_2024': 'Other'},
    {'Antibiotic': 'Netilmicin', 'aware_2019': 'Watch', 'aware_2024': 'Other'},
    {'Antibiotic': 'Roxithromycin', 'aware_2019': 'Watch', 'aware_2024': 'Other'},
    {'Antibiotic': 'Sodium fusidate', 'aware_2019': 'Watch', 'aware_2024': 'Watch'},
    {'Antibiotic': 'Cefepime + Enmetazobactam', 'aware_2019': '?', 'aware_2024': '?'}
]

# Step 2: Convert to DataFrame
vtm_manual_additions = pd.DataFrame(vtm_manual_additions)

# Step 3: Add required columns to match vtm_match_cleaned
vtm_manual_additions['nm'] = vtm_manual_additions['Antibiotic']
vtm_manual_additions['id'] = pd.NA
vtm_manual_additions['atc_route'] = pd.NA

# Step 4: Reorder columns
vtm_manual_additions = vtm_manual_additions[vtm_match_cleaned.columns]

# Step 5: Concatenate and assign to a new cleaned version
vtm_clean_extended = pd.concat([vtm_match_cleaned, vtm_manual_additions], ignore_index=True)

# Set Ampicillin + Flucloxacillin category to match Co-fluampicil
vtm_clean_extended.loc[
    vtm_clean_extended['Antibiotic'] == 'Ampicillin + Flucloxacillin',
    ['aware_2019', 'aware_2024']
] = 'Other'

vtm_clean_extended

/tmp/ipykernel_419/1384974639.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  vtm_clean_extended = pd.concat([vtm_match_cleaned, vtm_manual_additions], ignore_index=True)


,Antibiotic,nm,id,atc_route,aware_2019,aware_2024
0,Amikacin,Amikacin liposomal,782421008.0,NaN,Watch,Watch
1,Amikacin,Amikacin,774534000.0,NaN,Watch,Watch
2,Amoxicillin,Amoxicillin,774586009.0,NaN,Access,Access
3,Ampicillin,Ampicillin + Flucloxacillin,774592003.0,NaN,Access,Access
4,Ampicillin,Ampicillin,774590006.0,NaN,Access,Access
...,...,...,...,...,...,...
129,Ticarcillin + Clavulanic acid,Ticarcillin + Clavulanic acid,NaN,NaN,Other,Other
130,Netilmicin,Netilmicin,NaN,NaN,Watch,Other
131,Roxithromycin,Roxithromycin,NaN,NaN,Watch,Other
132,Sodium fusidate,Sodium fusidate,NaN,NaN,Watch,Watch


We can upload the resulting table to BigQuery as table 'chris.aware_list_extended'

We can adjust our VTM_method SQL query to match this new list

In [43]:
sql = f"""
WITH matched_vtms AS ( -- This query does the fuzzy matching of Antibiotic name to VTM name
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    vtm.id,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list_extended` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
),

duplicate_vtms AS ( -- This produces a list of duplicate rows where a direct match already exists
  SELECT *
  FROM matched_vtms
  WHERE nm IN (
    SELECT nm
    FROM matched_vtms
    WHERE atc_route IS NULL
    GROUP BY nm
    HAVING COUNT(*) > 1
  )
  AND match_type = 'ASSUMED'
),

vmp_with_route AS (
  SELECT DISTINCT
    vmp.id AS id,
    vmp.vtm AS vtm,
    vmp.nm AS nm,
    COALESCE(sroute.route, routelookup.who_route) AS vmp_atc_route
  FROM `ebmdatalab.dmd.vmp` vmp
  LEFT JOIN dmd.ont ont ON vmp.id = ont.vmp
  LEFT JOIN dmd.ontformroute ofr ON ont.form = ofr.cd
  LEFT JOIN chris.vmp_single_route_identifier sroute ON vmp.id = sroute.vmp_id
  LEFT JOIN `scmd_dmd_views.dmd_to_atc_route` routelookup ON ofr.descr = routelookup.dmd_ofr
  WHERE ofr.descr NOT LIKE '%.auricular'
  AND ofr.descr NOT LIKE '%.cutaneous'
  AND ofr.descr NOT LIKE '%.ophthalmic'
  AND ofr.descr NOT LIKE '%.oromucosal'
  AND ofr.descr NOT LIKE '%.intralesional'
  AND ofr.descr NOT LIKE '%.nasal'
)

SELECT DISTINCT
  matched_vtms.Antibiotic,
  matched_vtms.nm as vtm_nm,
  matched_vtms.id as vtm_id,
  vmp.nm as vmp_nm,
  vmp.id as vmp_id,
  vmp.vmp_atc_route,
  matched_vtms.atc_route,
  matched_vtms.aware_2019,
  matched_vtms.aware_2024
FROM matched_vtms
LEFT JOIN vmp_with_route vmp
  ON matched_vtms.id = vmp.vtm
  AND (
    matched_vtms.atc_route IS NULL
    OR matched_vtms.atc_route = vmp.vmp_atc_route
  )
WHERE NOT EXISTS ( -- removes the duplicates identified above
  SELECT 1
  FROM duplicate_vtms
  WHERE matched_vtms.Antibiotic = duplicate_vtms.Antibiotic AND matched_vtms.nm = duplicate_vtms.nm
) AND vmp.id IS NOT NULL
ORDER BY matched_vtms.Antibiotic ASC;
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vmps_matched_extended.csv')

# Use the cached_read function from the bq library to run the query.
vmps_matched_extended = bq.cached_read(sql, csv_path=csv_path)

# Force vmp_id and vtm_id to proper string type
vmps_matched_extended['vmp_id'] = vmps_matched_extended['vmp_id'].astype('string')
vmps_matched_extended['vtm_id'] = vmps_matched_extended['vtm_id'].astype('string')

display(vmps_matched_extended)

,Antibiotic,vtm_nm,vtm_id,vmp_nm,vmp_id,vmp_atc_route,atc_route,aware_2019,aware_2024
0,Amikacin,Amikacin liposomal,782421008,Amikacin liposomal 590mg nebuliser dispersion ...,39601711000001109,Inhal.solution,NaN,Watch,Watch
1,Amikacin,Amikacin,774534000,Amikacin 500mg/2ml solution for injection ampo...,44422511000001103,P,NaN,Watch,Watch
2,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection ampoules,44007311000001107,NaN,NaN,Watch,Watch
3,Amikacin,Amikacin,774534000,Amikacin 500mg/100ml infusion polyethylene bot...,41823511000001108,P,NaN,Watch,Watch
4,Amikacin,Amikacin,774534000,Amikacin 400micrograms/0.1ml intravitreal inje...,38244211000001108,NaN,NaN,Watch,Watch
...,...,...,...,...,...,...,...,...,...
613,Vancomycin,Vancomycin,777914002,Vancomycin 250mg/5ml oral suspension,8818711000001109,O,NaN,Watch,Watch
614,Vancomycin,Vancomycin,777914002,Vancomycin 50mg/5ml oral solution,12940811000001109,O,NaN,Watch,Watch
615,Vancomycin,Vancomycin,777914002,Vancomycin 62.5mg/5ml oral solution,13455811000001106,O,NaN,Watch,Watch
616,Vancomycin,Vancomycin,777914002,Vancomycin 125mg/5ml oral solution,15453011000001107,O,NaN,Watch,Watch


### Comparing VMPs

We now have a list of VMPs from the VTM method. We can compare these against the UKHSA supplied list to ensure we capture everything.

In [44]:
vmps_matched_extended[['vmp_id', 'vmp_nm', 'aware_2024']]

,vmp_id,vmp_nm,aware_2024
0,39601711000001109,Amikacin liposomal 590mg nebuliser dispersion ...,Watch
1,44422511000001103,Amikacin 500mg/2ml solution for injection ampo...,Watch
2,44007311000001107,Amikacin 25mg/5ml solution for injection ampoules,Watch
3,41823511000001108,Amikacin 500mg/100ml infusion polyethylene bot...,Watch
4,38244211000001108,Amikacin 400micrograms/0.1ml intravitreal inje...,Watch
...,...,...,...
613,8818711000001109,Vancomycin 250mg/5ml oral suspension,Watch
614,12940811000001109,Vancomycin 50mg/5ml oral solution,Watch
615,13455811000001106,Vancomycin 62.5mg/5ml oral solution,Watch
616,15453011000001107,Vancomycin 125mg/5ml oral solution,Watch


In [45]:
ukhsa_filtered = ukhsa_original_matched_analysis[
    (ukhsa_original_matched_analysis['vmp_id'] != 0) &
    (ukhsa_original_matched_analysis['Include'] == 1)
][['vmp_id', 'vmp_nm', 'UK_AWaRe_2024']]
ukhsa_filtered

,vmp_id,vmp_nm,UK_AWaRe_2024
0,35899811000001104,Amikacin 100mg/2ml solution for injection vials,w
5,35111911000001104,Amikacin 25mg/5ml solution for injection vials,w
6,41823511000001108,Amikacin 500mg/100ml infusion polyethylene bot...,w
7,35899911000001109,Amikacin 500mg/2ml solution for injection vials,w
8,44422511000001103,Amikacin 500mg/2ml solution for injection ampo...,w
...,...,...,...
941,35049111000001102,Vancomycin 50mg/10ml solution for injection pr...,w
943,12940811000001109,Vancomycin 50mg/5ml oral solution,w
944,21278311000001101,Vancomycin 5mg/0.5ml solution for injection pr...,w
945,13455811000001106,Vancomycin 62.5mg/5ml oral solution,w


In [46]:
# Step 1: Rename columns for consistency
ukhsa_df = ukhsa_filtered.rename(columns={'UK_AWaRe_2024': 'ukhsa_aware_2024'})
vmps_df = vmps_matched_extended.rename(columns={'aware_2024': 'vmps_aware_2024'})

# Step 2: Perform outer merge on vmp_id only
merged_vmps = pd.merge(
    ukhsa_df,
    vmps_df,
    on='vmp_id',
    how='outer',
    indicator=True
)

# Step 3: Label the source of each row
merged_vmps['_merge'] = merged_vmps['_merge'].replace({
    'left_only': 'ukhsa_list_only',
    'right_only': 'vtm_method_ext_only',
    'both': 'both'
})

/tmp/ipykernel_419/189286236.py:15: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  merged_vmps['_merge'] = merged_vmps['_merge'].replace({


In [47]:
merged_vmps_differences = merged_vmps[merged_vmps['_merge'] != 'both']
merged_vmps_differences = merged_vmps_differences.sort_values(by='_merge')

In [48]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(merged_vmps_differences)

,vmp_id,vmp_nm_x,ukhsa_aware_2024,Antibiotic,vtm_nm,vtm_id,vmp_nm_y,vmp_atc_route,atc_route,aware_2019,vmps_aware_2024,_merge
241,32936911000001103,Colistimethate 2million unit powder for nebuliser solution unit dose vials,r,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,ukhsa_list_only
645,7526011000001106,Flucloxacillin 125mg / 5ml oral solution sugar free,a,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,ukhsa_list_only
195,22399111000001107,Vancomycin 20mg pastilles,w,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,ukhsa_list_only
203,22555311000001104,"Colistimethate 1,662,500unit inhalation powder capsules",r,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,ukhsa_list_only
607,4256011000001107,Colistimethate 1million unit powder for nebuliser solution unit dose vials,r,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,ukhsa_list_only
1,10150911000001107,NaN,NaN,Benzathine benzylpenicillin,Benzathine benzylpenicillin,1234764000,"Benzathine benzylpenicillin 600,000unit powder and solvent for suspension for injection vials",P,NaN,Access,Access,vtm_method_ext_only
429,41882411000001108,NaN,NaN,Telavancin,Telavancin,777693006,Telavancin 750mg powder for solution for infusion vials,P,NaN,Reserve,Reserve,vtm_method_ext_only
428,41882311000001101,NaN,NaN,Telavancin,Telavancin,777693006,Telavancin 250mg powder for solution for infusion vials,P,NaN,Reserve,Reserve,vtm_method_ext_only
416,41742911000001107,NaN,NaN,Metronidazole,Metronidazole,776774005,Metronidazole 25% oromucosal gel sugar free,NaN,NaN,Access,Access,vtm_method_ext_only
415,41742711000001105,NaN,NaN,Metronidazole,Metronidazole,776774005,Metronidazole 0.75% vaginal gel,V,NaN,Access,Access,vtm_method_ext_only


Of the remaining discrepancies:  
**ukhsa_list_only**  
- Colistimethate nebuliser solution/inhalation capsules - the UKHSA online list specifies only Colistin IV and Colistin oral.  
- Vancomycin 20mg pastilles - these have been excluded from our list as oromucosal so presumably for localised infection.  
  
**vtm_method_ext_only**  
- Metronidazole 25% oromucosal gel sugar free, metronidazole 0.75% vaginal gel, clindamycin 2% vaginal cream - these have presumably been excluded from the UKHSA list as for localised infection.  
- The remainder of unmatched VMPs here seem to be uncommonly used products.  

We can refine our SQL code to exclude the oromucosal gel and vaginal gel/cream:

In [49]:
sql = f"""
WITH matched_vtms AS ( -- This query does the fuzzy matching of Antibiotic name to VTM name
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    vtm.id,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list_extended` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
),

duplicate_vtms AS ( -- This produces a list of duplicate rows where a direct match already exists
  SELECT *
  FROM matched_vtms
  WHERE nm IN (
    SELECT nm
    FROM matched_vtms
    WHERE atc_route IS NULL
    GROUP BY nm
    HAVING COUNT(*) > 1
  )
  AND match_type = 'ASSUMED'
),

vmp_with_route AS (
  SELECT DISTINCT
    vmp.id AS id,
    vmp.vtm AS vtm,
    vmp.nm AS nm,
    COALESCE(sroute.route, routelookup.who_route) AS vmp_atc_route
  FROM `ebmdatalab.dmd.vmp` vmp
  LEFT JOIN dmd.ont ont ON vmp.id = ont.vmp
  LEFT JOIN dmd.ontformroute ofr ON ont.form = ofr.cd
  LEFT JOIN chris.vmp_single_route_identifier sroute ON vmp.id = sroute.vmp_id
  LEFT JOIN `scmd_dmd_views.dmd_to_atc_route` routelookup ON ofr.descr = routelookup.dmd_ofr
  WHERE ofr.descr NOT LIKE '%.auricular'
  AND ofr.descr NOT LIKE '%.cutaneous'
  AND ofr.descr NOT LIKE '%.ophthalmic'
  AND ofr.descr NOT LIKE '%.oromucosal'
  AND ofr.descr NOT LIKE '%.intralesional'
  AND ofr.descr NOT LIKE '%.nasal'
  AND ofr.descr NOT LIKE '%.gingival'
  AND ofr.descr != 'gel.vaginal'
  AND ofr.descr != 'cream.vaginal'
)

SELECT DISTINCT
  matched_vtms.Antibiotic,
  matched_vtms.nm as vtm_nm,
  matched_vtms.id as vtm_id,
  vmp.nm as vmp_nm,
  vmp.id as vmp_id,
  vmp.vmp_atc_route,
  matched_vtms.atc_route,
  matched_vtms.aware_2019,
  matched_vtms.aware_2024
FROM matched_vtms
LEFT JOIN vmp_with_route vmp
  ON matched_vtms.id = vmp.vtm
  AND (
    matched_vtms.atc_route IS NULL
    OR matched_vtms.atc_route = vmp.vmp_atc_route
  )
WHERE NOT EXISTS ( -- removes the duplicates identified above
  SELECT 1
  FROM duplicate_vtms
  WHERE matched_vtms.Antibiotic = duplicate_vtms.Antibiotic AND matched_vtms.nm = duplicate_vtms.nm
) AND vmp.id IS NOT NULL
ORDER BY matched_vtms.Antibiotic ASC;
"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'vmps_matched_extended.csv')

# Use the cached_read function from the bq library to run the query.
vmps_matched_extended = bq.cached_read(sql, csv_path=csv_path)

# Force vmp_id and vtm_id to proper string type
vmps_matched_extended['vmp_id'] = vmps_matched_extended['vmp_id'].astype('string')
vmps_matched_extended['vtm_id'] = vmps_matched_extended['vtm_id'].astype('string')

display(vmps_matched_extended)

Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


,Antibiotic,vtm_nm,vtm_id,vmp_nm,vmp_id,vmp_atc_route,atc_route,aware_2019,aware_2024
0,Amikacin,Amikacin,774534000,Amikacin 500mg/2ml solution for injection vials,35899911000001109,P,None,Watch,Watch
1,Amikacin,Amikacin,774534000,Amikacin 100mg/2ml solution for injection vials,35899811000001104,P,None,Watch,Watch
2,Amikacin,Amikacin,774534000,Amikacin 500mg/2ml solution for injection ampo...,44422511000001103,P,None,Watch,Watch
3,Amikacin,Amikacin,774534000,Amikacin 500mg/100ml infusion polyethylene bot...,41823511000001108,P,None,Watch,Watch
4,Amikacin,Amikacin,774534000,Amikacin 25mg/5ml solution for injection pre-f...,35104511000001105,None,None,Watch,Watch
...,...,...,...,...,...,...,...,...,...
610,Vancomycin,Vancomycin,777914002,Vancomycin 1mg/0.1ml solution for injection pr...,21277311000001102,None,None,Watch,Watch
611,Vancomycin,Vancomycin,777914002,Vancomycin 5mg/0.5ml solution for injection pr...,21278311000001101,None,None,Watch,Watch
612,Vancomycin,Vancomycin,777914002,Vancomycin 2.5mg/0.5ml solution for injection ...,28047511000001105,None,None,Watch,Watch
613,Vancomycin,Vancomycin,777914002,Vancomycin 2mg/0.1ml intravitreal injection kit,40529511000001103,None,None,Watch,Watch


<a id='s5'></a>
## Section 5: Summary
We have created a method using SQL logic to produce a list of VMPs categorised with UK AWaRe categorisation.  

We have validated the method through comparison against a UKHSA provided list and extended / refined where needed.

### Final SQL code

In [50]:
sql = f"""
WITH matched_vtms AS ( -- This query does the fuzzy matching of Antibiotic name to VTM name
  SELECT  
    aware.Antibiotic,
    aware.atc_route,
    aware.aware_2019,
    aware.aware_2024,
    vtm.nm,
    vtm.id,
    CASE
      WHEN aware.Antibiotic = vtm.nm THEN 'DIRECT'
      ELSE 'ASSUMED'
    END AS match_type
  FROM `ebmdatalab.chris.aware_list_extended` aware
  LEFT JOIN `ebmdatalab.dmd.vtm` vtm 
    ON vtm.nm LIKE CONCAT('%', aware.Antibiotic, '%')
  WHERE vtm.invalid IS NOT TRUE
),

duplicate_vtms AS ( -- This produces a list of duplicate rows where a direct match already exists
  SELECT *
  FROM matched_vtms
  WHERE nm IN (
    SELECT nm
    FROM matched_vtms
    WHERE atc_route IS NULL
    GROUP BY nm
    HAVING COUNT(*) > 1
  )
  AND match_type = 'ASSUMED'
),

vmp_with_route AS (
  SELECT DISTINCT
    vmp.id AS id,
    vmp.vtm AS vtm,
    vmp.nm AS nm,
    COALESCE(sroute.route, routelookup.who_route) AS vmp_atc_route
  FROM `ebmdatalab.dmd.vmp` vmp
  LEFT JOIN dmd.ont ont ON vmp.id = ont.vmp
  LEFT JOIN dmd.ontformroute ofr ON ont.form = ofr.cd
  LEFT JOIN chris.vmp_single_route_identifier sroute ON vmp.id = sroute.vmp_id
  LEFT JOIN `scmd_dmd_views.dmd_to_atc_route` routelookup ON ofr.descr = routelookup.dmd_ofr
  WHERE ofr.descr NOT LIKE '%.auricular'
  AND ofr.descr NOT LIKE '%.cutaneous'
  AND ofr.descr NOT LIKE '%.ophthalmic'
  AND ofr.descr NOT LIKE '%.oromucosal'
  AND ofr.descr NOT LIKE '%.intralesional'
  AND ofr.descr NOT LIKE '%.nasal'
  AND ofr.descr NOT LIKE '%.gingival'
  AND ofr.descr != 'gel.vaginal'
  AND ofr.descr != 'cream.vaginal'
)

SELECT DISTINCT
  matched_vtms.Antibiotic,
  matched_vtms.nm as vtm_nm,
  matched_vtms.id as vtm_id,
  vmp.nm as vmp_nm,
  vmp.id as vmp_id,
  vmp.vmp_atc_route,
  matched_vtms.atc_route,
  matched_vtms.aware_2019,
  matched_vtms.aware_2024
FROM matched_vtms
LEFT JOIN vmp_with_route vmp
  ON matched_vtms.id = vmp.vtm
  AND (
    matched_vtms.atc_route IS NULL
    OR matched_vtms.atc_route = vmp.vmp_atc_route
  )
WHERE NOT EXISTS ( -- removes the duplicates identified above
  SELECT 1
  FROM duplicate_vtms
  WHERE matched_vtms.Antibiotic = duplicate_vtms.Antibiotic AND matched_vtms.nm = duplicate_vtms.nm
) AND vmp.id IS NOT NULL
ORDER BY matched_vtms.Antibiotic ASC;
"""

### Outstanding questions

The UKHSA online list specifies only Colistin IV and Colistin oral. Should Colistin inhalation be included?
Cefepime + Enmetazobactam isn't included in the UKHSA list. Should it be included for potential future use? With what categorisation?
Vancomycin 20mg pastilles - these have been excluded from our list as oromucosal so presumably for localised infection. Should they be included?